# 품질 검수 E2E 프로덕션 검증 노트북

**목적**: `quality_check_worker_example.py`가 실제로 도는 것과 *정확히 같은 프로덕션 코드*
(`worker/models/sam3/`, `worker/models/quality_vlm/`)를 그대로 import해서, 단계별로
직접 실행하며 눈으로 확인한다. `AI/validation/QUAL/`의 기존 스파이크 노트북들과 달리
**로직을 복제하지 않는다** — 여기서 쓰는 함수는 워커가 쓰는 것과 한 글자도 다르지 않다.

확인하려는 것 3가지:
1. **단계별 지연시간** — 세그멘테이션/항목화/오버레이/VLM설명/임베딩 각각 몇 ms인지,
   그리고 첫 요청(콜드, VLM 최초 로드 포함)과 두 번째 요청(웜)이 얼마나 다른지.
2. **백엔드로 전송되는 결과의 정확한 형태** — `run_inference()`가 게이트웨이에
   `POST /workers/result`로 보내는 것과 100% 같은 JSON을 그대로 조립해서 보여준다.
3. **중간 산출물 시각화** — 세그멘테이션 마스크, attention map, 넘버링 bbox 이미지를
   `plt.imshow`로 그때그때 확인한다.

## 실행 방법

- **반드시 이 노트북을 `worker/` 디렉터리에서 연다** (Jupyter 작업 디렉터리가 `worker/`여야
  `sys.path` 설정이 워커 스크립트와 동일하게 맞는다).
- 워커를 돌릴 때 쓰던 것과 **같은 conda 환경**에서 커널을 띄운다(예: `qual`).
- 아래 환경변수를 워커 실행 때와 동일하게 맞춘다(첫 코드 셀에서 override 가능):
  - `QUALITY_BACKEND` — `mock` 또는 `sam3_pcs`
  - `QUALITY_VLM_BACKEND` — `mock` 또는 `qwen3_vl`
  - `GATEWAY_URL`/`WORKER_API_KEY` — 있으면 실제 게이트웨이에 오버레이 이미지를 업로드해서
    받은 URL을 그대로 다시 받아와 보여준다(진짜 왕복 확인). 없으면 로컬 디스크에 저장하는
    dry-run으로 자동 대체된다.
- `TEST_PAYLOAD`의 `image_urls`를 실제 검수해보고 싶은 이미지(HTTP(S) URL 또는 로컬 파일
  경로 — `load_source_image()`가 둘 다 받는다)로 바꾼다.

실행 순서: 0(설정) → 1(세그멘터 로드) → 2(프롬프트셋) → 3(세그멘테이션) → 4(항목화) →
5(점수/요약) → 6(오버레이) → 7(VLM 설명) → 8(임베딩) → 9(최종 result 조립) →
10(지연시간 요약) → 11(웜/콜드 비교, 재사용 함수).

In [ ]:
import json
import os
import sys
import time

import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

# `quality_check_worker_example.py`와 완전히 같은 sys.path 설정 — 이래야 같은 모듈을
# 같은 방식(`sam3.xxx`/`quality_vlm.xxx` 네임스페이스)으로 import한다.
# Jupyter의 cwd는 보통 노트북 파일이 있는 폴더다 — worker/에서 열면 cwd=worker/,
# worker/validation/에서 열면 cwd=worker/validation/이 된다. 둘 다 대응한다.
from pathlib import Path

_CWD = Path(os.getcwd())
_WORKER_DIR = _CWD if (_CWD / "models").is_dir() else _CWD.parent
_MODELS_DIR = str(_WORKER_DIR / "models")
assert os.path.isdir(_MODELS_DIR), (
    f"'{_MODELS_DIR}'가 없습니다 — 이 노트북은 worker/ 또는 worker/validation/에서 열어야 합니다."
)
sys.path.insert(0, os.getenv("QUALITY_MODEL_CODE_PATH", _MODELS_DIR))
# quality_check_worker_example.py는 스크립트 자기 위치(worker/)가 sys.path에 자동으로
# 잡혀서 inspection_logger를 바로 import한다 — 노트북은 그 자동 추가가 없어서(cwd 기준
# 암묵적 경로만 잡힘) worker/ 자체도 명시적으로 넣어줘야 한다.
sys.path.insert(0, str(_WORKER_DIR))

from quality_vlm import registry as vlm_registry  # noqa: E402
from quality_vlm.describer import attach_descriptions, build_describer  # noqa: E402
from sam3 import registry  # noqa: E402
from sam3.contracts import BaseColor, GarmentCategory  # noqa: E402
from sam3.findings import extract_findings  # noqa: E402
from sam3.garment_embedding import (  # noqa: E402
    EMBEDDING_MODEL_ID,
    GarmentEmbeddingNotReady,
    embed_garment,
)
from sam3.overlay import build_annotated_images, build_attention_maps  # noqa: E402
from sam3.prompts import select_prompt_set  # noqa: E402
from sam3.scoring import build_notes, score_findings, summarize, summarize_by_image  # noqa: E402
from sam3.segmenter import InspectionTarget, build_segmenter  # noqa: E402

from inspection_logger import log_inspection_result  # noqa: E402

print("production 모듈 import 완료 — quality_check_worker_example.py와 동일한 코드 경로")

## 0. 백엔드·입력 설정

워커 프로세스를 이미 `QUALITY_BACKEND=...`로 띄워둔 셸에서 Jupyter를 실행했다면 환경변수를
상속받아 아래에서 그대로 읽힌다. 다른 값으로 바꿔 테스트하고 싶으면 `os.environ[...]`을
이 셀에서 먼저 덮어쓴다.

In [ ]:
# 필요하면 여기서 override (워커 실행 때와 다르게 테스트하고 싶을 때만):
# os.environ["QUALITY_BACKEND"] = "sam3_pcs"
# os.environ["QUALITY_VLM_BACKEND"] = "qwen3_vl"

QUALITY_BACKEND = os.getenv("QUALITY_BACKEND", registry.ACTIVE_SEGMENTER)
QUALITY_VLM_BACKEND = os.getenv("QUALITY_VLM_BACKEND", vlm_registry.ACTIVE_VLM_BACKEND)
GATEWAY_URL = os.getenv("GATEWAY_URL", "").rstrip("/")
WORKER_API_KEY = os.getenv("WORKER_API_KEY")

print(f"QUALITY_BACKEND      = {QUALITY_BACKEND}")
print(f"QUALITY_VLM_BACKEND  = {QUALITY_VLM_BACKEND}")
if GATEWAY_URL and WORKER_API_KEY:
    print(f"GATEWAY_URL          = {GATEWAY_URL}  (실제 업로드 사용)")
else:
    print("GATEWAY_URL/WORKER_API_KEY 없음 -> 오버레이 이미지는 로컬 디스크에 dry-run 저장")

# `app/quality/schemas.py`의 InspectionRequest와 필드가 동일하다 — BE가 실제로 보내는 payload.
# image_urls를 실제 확인하고 싶은 이미지(HTTP(S) URL 또는 로컬 파일 경로)로 바꿀 것.
TEST_PAYLOAD = {
    "product_id": 999001,  # 테스트용 — 실제 상품 id와 겹치지 않게 임의로 잡음
    "image_urls": [
        # _WORKER_DIR 기준 절대경로 — cwd가 worker/든 worker/validation/든 안전하다.
        # sample_images/는 로컬 전용(.gitignore) 실제 상품 사진 — registry.py 주석의
        # 실측 중복 케이스(small_pro.jpg)로 우선 검증한다.
        str(_WORKER_DIR / "validation" / "sample_images" / "small_pro.jpg"),
    ],
    "category": "TOP_SHIRT",
    "base_color": "LIGHT",
    "include_attention_maps": True,
    "include_descriptions": True,
}

step_timings_ms: dict[str, float] = {}


def record(name: str, started: float) -> float:
    """경과시간(ms)을 기록하고 즉시 출력한다."""
    elapsed = round((time.perf_counter() - started) * 1000, 2)
    step_timings_ms[name] = elapsed
    print(f"[{name}] {elapsed} ms")
    return elapsed


def notebook_upload(image: Image.Image, prefix: str) -> str:
    """실 게이트웨이가 있으면 storage_utils.upload_image()를 그대로 쓴다(프로덕션과 동일
    경로) — 없으면 로컬에 저장하고 file:// 경로를 돌려주는 dry-run으로 대체한다."""
    if GATEWAY_URL and WORKER_API_KEY:
        from storage_utils import upload_image

        return upload_image(image, prefix)
    out_dir = os.path.join(os.getcwd(), "_e2e_dry_run_uploads")
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{prefix}_{len(os.listdir(out_dir))}.png")
    image.save(path)
    print(f"[dry-run] {prefix} 로컬 저장: {path}")
    return f"file://{os.path.abspath(path)}"

## 1. 세그멘터 로드

프로덕션에서는 `_ensure_loaded()`가 워커 프로세스당 한 번만 호출한다(job마다 새로 만들지
않음). 이 로드 시간은 실제 job의 `latency_ms`에는 잡히지 않는다 — `run_inference()`의
`started = time.perf_counter()`가 로드 *이후*에 찍히기 때문이다. 아래도 그와 똑같이
로드를 먼저 끝내고, 그다음 셀부터 인퍼런스 타이머를 시작한다.

In [ ]:
_t0 = time.perf_counter()
segmenter_config = registry.segmenter_config(QUALITY_BACKEND)
segmenter = build_segmenter(QUALITY_BACKEND)
record("[로드] 세그멘터", _t0)
print(f"model_id={segmenter_config['model_id']}")

# 인퍼런스 타이머 시작 — 프로덕션 run_inference()의 `started`와 같은 지점.
inference_started = time.perf_counter()

## 2. 프롬프트셋 선택

In [ ]:
category = GarmentCategory(TEST_PAYLOAD["category"])
base_color = BaseColor(TEST_PAYLOAD.get("base_color", BaseColor.light.value))
prompt_set = select_prompt_set(category, base_color)

target = InspectionTarget(
    product_id=int(TEST_PAYLOAD["product_id"]),
    image_urls=tuple(TEST_PAYLOAD["image_urls"]),
    prompt_set=prompt_set,
)
print(f"prompt_set.key={prompt_set.key}, defect_prompts={len(prompt_set.defect_prompts)}개")

## 3. 세그멘테이션 (이미지별 SAM3 추론)

In [ ]:
results = []
for i, url in enumerate(TEST_PAYLOAD["image_urls"]):
    _t0 = time.perf_counter()
    result = segmenter.segment(target, i)
    record(f"세그멘테이션[{i}]", _t0)
    print(
        f"  image[{i}] garment_mask={'있음' if result.garment_mask is not None else '없음'}, "
        f"instances={len(result.instances)}, benign={len(result.benign_instances)}"
    )
    results.append(result)

In [ ]:
# 시각화 — 원본 위에 의류 마스크를 반투명하게 얹어서 확인한다.
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6), squeeze=False)
for ax, result in zip(axes[0], results):
    ax.imshow(result.base_image)
    if result.garment_mask is not None:
        ax.imshow(result.garment_mask, alpha=0.25, cmap="Greens")
    ax.set_title(f"image[{result.image_index}] instances={len(result.instances)}")
    ax.axis("off")
plt.show()

## 4. 하자 항목화 (extract_findings)

의류 겹침 필터 → 양성 억제 → same-type/cross-type 중복 병합을 거쳐 `Finding` 리스트로
승격한다(CH010).

In [ ]:
_t0 = time.perf_counter()
findings_result = extract_findings(results, prompt_set, segmenter_config)
record("하자 항목화(extract_findings)", _t0)

print(f"findings={len(findings_result.findings)}, section_scores={len(findings_result.section_scores)}")
for f in findings_result.findings:
    print(
        f"  - no={f.no} {f.label} status={f.status.value} severity={f.severity.value} "
        f"conf={f.confidence:.3f} image={f.image_index}"
    )

## 5. 점수·등급·요약 텍스트

In [ ]:
_t0 = time.perf_counter()
condition_score, condition_grade = score_findings(findings_result.findings)
notes = build_notes(findings_result.findings, findings_result.unreliable_prompts)
summary_text = summarize(findings_result.findings)
image_summaries = summarize_by_image(findings_result.findings, len(TEST_PAYLOAD["image_urls"]))
record("점수/등급/요약", _t0)

print(f"condition_score={condition_score}, condition_grade={condition_grade.value}")
print(f"summary: {summary_text}")
print(f"notes: {notes}")

## 6. 오버레이 이미지 — attention map + 넘버링 bbox

`include_attention_maps=False`면 BE 기본 계약처럼 attention_maps는 빈 리스트로 둔다.
넘버링 bbox 이미지(`annotated_images`)는 CH007 결정에 따라 옵션이 아니라 항상 만든다.

In [ ]:
attention_maps = []
if TEST_PAYLOAD.get("include_attention_maps"):
    _t0 = time.perf_counter()
    attention_maps = build_attention_maps(
        results, findings_result.kept_instances_by_image, findings_result.findings, notebook_upload
    )
    record("attention_maps 생성+업로드", _t0)

_t0 = time.perf_counter()
annotated_images = build_annotated_images(results, findings_result.findings, notebook_upload)
record("annotated_images(넘버링 bbox) 생성+업로드", _t0)

for m in annotated_images:
    print(f"annotated image[{m.image_index}] -> {m.url}")
for m in attention_maps:
    print(f"attention {m.defect_type.value} image[{m.image_index}] -> {m.url}")

In [ ]:
# 업로드된(또는 dry-run 저장된) 이미지를 다시 받아와 눈으로 확인한다 — 실제 게이트웨이를
# 썼다면 URL 왕복(업로드->공개 URL->재다운로드)까지 확인하는 것이다.
def _load_for_display(url: str) -> Image.Image:
    if url.startswith("http"):
        from storage_utils import download_image

        return download_image(url)
    return Image.open(url.replace("file://", ""))


to_show = [(m.url, f"annotated[{m.image_index}]") for m in annotated_images]
to_show += [(m.url, f"attention[{m.image_index}] {m.defect_type.value}") for m in attention_maps]

if to_show:
    fig, axes = plt.subplots(1, len(to_show), figsize=(6 * len(to_show), 6), squeeze=False)
    for ax, (url, title) in zip(axes[0], to_show):
        ax.imshow(_load_for_display(url))
        ax.set_title(title)
        ax.axis("off")
    plt.show()
else:
    print("표시할 오버레이 이미지 없음(하자 미검출)")

## 7. 하자 설명(VLM, 옵션)

`_ensure_vlm_loaded()`는 프로덕션에서도 **job 처리 도중**(첫 `include_descriptions=true`
요청 때) 로드된다 — 즉 이 셀의 첫 실행 시간에는 Qwen3-VL 콜드 로드가 그대로 포함된다.
11장에서 같은 파이프라인을 한 번 더 돌려 웜 상태와 비교한다.

In [ ]:
findings_with_desc = findings_result.findings
describer = None
if TEST_PAYLOAD.get("include_descriptions"):
    _t0 = time.perf_counter()
    try:
        describer = build_describer(QUALITY_VLM_BACKEND)
    except Exception as exc:
        print(f"VLM describer 로드 실패, mock으로 대체: {exc}")
        describer = build_describer("mock")
    record("[로드] VLM describer(첫 사용 시 콜드 포함)", _t0)

    _t0 = time.perf_counter()
    findings_with_desc = attach_descriptions(results, findings_result.findings, describer)
    record("하자 설명 생성(attach_descriptions)", _t0)

    for f in findings_with_desc:
        if f.description:
            print(f"  - [{f.label}] {f.description}")
else:
    print("include_descriptions=False -> 건너뜀")

## 8. 추천용 임베딩 (best-effort)

SAM3가 하자 탐지 전에 이미 만든 의류 크롭을 재사용한다(새 세그멘테이션 없음). mock
백엔드는 `garment_crop`이 전부 `None`이라 자연히 건너뛴다. 실패해도 검수 결과 자체는
그대로 나가야 한다는 프로덕션 원칙을 그대로 재현한다.

In [ ]:
_t0 = time.perf_counter()
garment_embedding = None
for r in results:
    if r.garment_crop is None:
        continue
    try:
        garment_embedding = embed_garment(r.garment_crop)
        break
    except GarmentEmbeddingNotReady as exc:
        print(f"[garment_embedding] 준비 안 됨, 건너뜀: {exc}")
        break
    except Exception as exc:  # noqa: BLE001 - 임베딩 실패가 검수를 막으면 안 된다
        print(f"[garment_embedding] 계산 실패, 건너뜀: {exc}")
        break
record("추천용 임베딩(best-effort)", _t0)
print("embedding 계산됨" if garment_embedding is not None else "embedding 없음(mock 백엔드거나 실패)")

## 9. 최종 result 조립 — 백엔드로 전송되는 정확한 JSON 형태

`run_inference()`가 만드는 dict와 키·순서·타입을 전부 동일하게 맞췄다. 이게 그대로
게이트웨이 `POST /workers/result`의 body가 되고, BE가 `GET /ai/jobs/{job_id}`로 돌려주는
`result` 필드가 된다.

In [ ]:
latency_ms = round((time.perf_counter() - inference_started) * 1000, 3)

result = {
    "inspection_id": f"insp_{TEST_PAYLOAD['product_id']}_notebook",
    "product_id": int(TEST_PAYLOAD["product_id"]),
    "model": {
        "backend": QUALITY_BACKEND,
        "model_id": segmenter_config["model_id"],
        "prompt_set": prompt_set.key,
    },
    "condition_score": condition_score,
    "condition_grade": condition_grade.value,
    "summary": summary_text,
    "image_summaries": [s.to_dict() for s in image_summaries],
    "findings": [f.to_dict() for f in findings_with_desc],
    "section_scores": [s.to_dict() for s in findings_result.section_scores],
    "attention_maps": [m.to_dict() for m in attention_maps],
    "annotated_images": [m.to_dict() for m in annotated_images],
    "notes": notes,
    "latency_ms": latency_ms,
}

if garment_embedding is not None:
    result["garment_embedding"] = garment_embedding
    result["garment_embedding_model_id"] = EMBEDDING_MODEL_ID
    result["garment_category"] = TEST_PAYLOAD["category"]

# 판단 근거 로컬 로그(CH008) — best-effort, 프로덕션과 동일하게 실패해도 결과는 그대로 낸다.
try:
    log_inspection_result(TEST_PAYLOAD, result)
except Exception as exc:  # noqa: BLE001
    print(f"[log_inspection_result] 실패(무시): {exc}")

print(json.dumps(result, indent=2, ensure_ascii=False))

## 10. 단계별 지연시간 요약

In [ ]:
print(f"{'단계':40s} {'ms':>10s}")
print("-" * 52)
for name, ms in step_timings_ms.items():
    print(f"{name:40s} {ms:>10.2f}")
print("-" * 52)
print(f"{'result.latency_ms (BE가 실제로 받는 값)':40s} {latency_ms:>10.2f}")
print(
    "\n참고: '[로드] 세그멘터'는 latency_ms에 안 잡힌다(프로세스 기동 시 1회, run_inference "
    "시작 전). 반면 '[로드] VLM describer'는 include_descriptions=true인 *첫* job의 "
    "latency_ms에 그대로 포함된다 — 워커가 그걸 job 처리 도중에 지연 로드하기 때문이다."
)

## 11. (옵션) 두 번째 요청으로 웜 latency 비교

세그멘터·describer 인스턴스를 재사용한 채로(재로딩 없음) 3~9단계를 한 번 더 돌려서,
"첫 요청(콜드, VLM 로드 포함)" vs "두 번째 요청(웜)"의 실제 차이를 확인한다. 워커
프로세스를 껐다 켜지 않는 한 실서비스에서도 두 번째 요청부터는 이 웜 시간이 기준이다.

In [ ]:
def run_once(payload: dict, *, segmenter, describer) -> dict:
    """3~9단계를 그대로 반복하는 헬퍼 — run_inference() 본문과 동일한 순서.
    시각화 없이 timing + 최종 result만 필요할 때(웜/콜드 비교, 다른 상품으로 재실행) 쓴다."""
    started = time.perf_counter()

    category = GarmentCategory(payload["category"])
    base_color = BaseColor(payload.get("base_color", BaseColor.light.value))
    prompt_set = select_prompt_set(category, base_color)
    target = InspectionTarget(
        product_id=int(payload["product_id"]),
        image_urls=tuple(payload["image_urls"]),
        prompt_set=prompt_set,
    )

    results = [segmenter.segment(target, i) for i in range(len(payload["image_urls"]))]
    findings_result = extract_findings(results, prompt_set, segmenter_config)
    condition_score, condition_grade = score_findings(findings_result.findings)
    notes = build_notes(findings_result.findings, findings_result.unreliable_prompts)
    summary_text = summarize(findings_result.findings)
    image_summaries = summarize_by_image(findings_result.findings, len(payload["image_urls"]))

    attention_maps = []
    if payload.get("include_attention_maps"):
        attention_maps = build_attention_maps(
            results, findings_result.kept_instances_by_image, findings_result.findings, notebook_upload
        )
    annotated_images = build_annotated_images(results, findings_result.findings, notebook_upload)

    findings_with_desc = findings_result.findings
    if payload.get("include_descriptions") and describer is not None:
        findings_with_desc = attach_descriptions(results, findings_result.findings, describer)

    return {
        "product_id": int(payload["product_id"]),
        "condition_score": condition_score,
        "condition_grade": condition_grade.value,
        "summary": summary_text,
        "image_summaries": [s.to_dict() for s in image_summaries],
        "findings": [f.to_dict() for f in findings_with_desc],
        "section_scores": [s.to_dict() for s in findings_result.section_scores],
        "attention_maps": [m.to_dict() for m in attention_maps],
        "annotated_images": [m.to_dict() for m in annotated_images],
        "notes": notes,
        "latency_ms": round((time.perf_counter() - started) * 1000, 3),
    }


warm_result = run_once(TEST_PAYLOAD, segmenter=segmenter, describer=describer)

print(f"콜드(1차, 위 9단계) latency_ms = {latency_ms}")
print(f"웜  (2차, 이 셀)   latency_ms = {warm_result['latency_ms']}")
print(
    f"차이 = {round(latency_ms - warm_result['latency_ms'], 2)} ms "
    "(양수면 콜드가 그만큼 더 느렸다는 뜻 — 대부분 VLM 최초 로드 때문)"
)

## 결론 (실행 후 채울 것)

- 콜드 vs 웜 latency_ms 차이:
- 단계별 병목(가장 오래 걸린 단계):
- BE/FE가 실제로 받는 JSON 형태에서 확인해야 할 이상 유무:

## 후속 작업

- 여러 product_id/이미지 조합으로 `run_once()`를 반복 호출해 latency 분포 확인
- 실 게이트웨이(`GATEWAY_URL`) 붙여서 업로드 왕복까지 포함한 latency 측정